In [1]:
# -*- coding: utf-8 -*-
"""
CTR Baseline — DCN-V2 + (DIEN or BST) [Notebook, OOM-safe, NaN-safe]
- 시퀀스: 해시 임베딩(고정 버킷), Lazy 파싱
- 본체: Deep & Cross v2(CrossNetMix: low-rank + experts) + Deep MLP
- 시퀀스 백본(선택):
    * DIEN: GRU → 은닉상태 시퀀스에 DIN-activation attention (가벼움, 최근성 잘 잡음)
    * BST : 얕은 Transformer(2~4층) + 위치임베딩, PAD 마스크 (긴 시퀀스 표현력 ↑)
- 범주형: train 기준 factorize(OOV), 연속형: BatchNorm1d
- 불균형: pos_weight, 지표: AUC/PR-AUC + EarlyStopping
- 안정성: fp16 마스킹 오버플로 방지, 전부 PAD 시퀀스 처리, NaN/±inf 정리
"""

import os, re, json, random, gc, warnings
warnings.filterwarnings("ignore")
import numpy as np, pandas as pd
from sklearn.metrics import roc_auc_score, average_precision_score
from sklearn.model_selection import train_test_split
import torch, torch.nn as nn, torch.nn.functional as F
from torch.utils.data import Dataset, DataLoader
from torch.cuda.amp import autocast, GradScaler

# =============== Utils ===============
def set_seed(seed=42):
    random.seed(seed); os.environ["PYTHONHASHSEED"]=str(seed)
    np.random.seed(seed); torch.manual_seed(seed); torch.cuda.manual_seed_all(seed)
    torch.backends.cudnn.deterministic=True; torch.backends.cudnn.benchmark=False

def infer_device(arg="auto"):
    if arg=="cpu": return torch.device("cpu")
    if arg=="cuda" or (arg=="auto" and torch.cuda.is_available()):
        print("Using CUDA"); return torch.device("cuda")
    print("Using CPU"); return torch.device("cpu")

def parse_seq_string(s: str):
    if s is None: return []
    s = str(s).strip()
    if not s or s.lower()=="nan": return []
    if s.startswith("[") and s.endswith("]"):
        s = s[1:-1]; toks = [t.strip().strip("'\"") for t in s.split(",")]
    else:
        s = re.sub(r"[^\d]+", ",", s); toks = [t for t in s.split(",") if t]
    out=[]
    for t in toks:
        try: out.append(int(t))
        except Exception:
            try: out.append(int(float(t)))
            except Exception: pass
    return out

def emb_dim_from_card(card, cap=64):
    return int(min(cap, max(8, round(1.6*(card**0.25)))))

# =============== Config ===============
class Args:
    train="./train.parquet"; test="./test.parquet"
    label="clicked"; seq_col="seq"; id_col="ID"

    sample_subset=1.0
    max_seq_len=50
    HASH_BUCKETS=262_144         # 2^18
    seq_emb_dim=64
    dropout=0.2

    # 시퀀스 백본: "dien" or "bst"
    seq_backbone="dien"
    # BST 전용
    bst_layers=2
    bst_nhead=4
    bst_ffn=128

    # DCN-V2 (CrossNetMix)
    cross_layers=3
    cross_low_rank=32
    cross_num_experts=4

    # Deep tower
    deep_units=[512,256,128]

    # Train
    epochs=3; batch_size=512; lr=1e-3; patience=2
    seed=42; device="auto"; num_workers=0; pin_memory=False
args=Args()
set_seed(args.seed); device=infer_device(args.device)

# =============== (옵션) 더미 데이터 생성 (파일 없을 때만) ===============
def create_dummy(num_rows, is_test=False):
    rng=np.random.default_rng(0)
    df=pd.DataFrame({
        "cont_feat1": rng.normal(size=num_rows),
        "cont_feat2": rng.random(num_rows)*10,
        "gender": rng.choice(["M","F","U"], num_rows, p=[0.45,0.45,0.1]),
        "age_group": rng.choice(["10s","20s","30s","40s+"], num_rows),
        "inventory_id": rng.integers(500, 520, num_rows),
        "day_of_week": rng.integers(0,7,num_rows),
        "hour": rng.integers(0,24,num_rows),
        "l_feat_14": rng.integers(1000,1050,num_rows),
    })
    seqs=[]
    for _ in range(num_rows):
        fmt=rng.choice(["pipe","comma","json","empty","single"])
        L=int(rng.integers(1,args.max_seq_len+10))
        items=[str(int(rng.integers(1000,1500))) for _ in range(L)]
        if fmt=="pipe": seqs.append("|".join(items))
        elif fmt=="comma": seqs.append(",".join(items))
        elif fmt=="json": seqs.append(f"[{','.join(items)}]")
        elif fmt=="empty": seqs.append(np.nan)
        else: seqs.append(items[0])
    df[args.seq_col]=seqs
    if is_test: df[args.id_col]=np.arange(num_rows)
    else: df[args.label]=rng.integers(0,2,num_rows)
    return df

if not os.path.exists(args.train):
    print("train.parquet 없음 → 더미 20k 생성"); create_dummy(20_000,False).to_parquet(args.train)
if not os.path.exists(args.test):
    print("test.parquet 없음 → 더미 5k 생성"); create_dummy(5_000,True).to_parquet(args.test)

# =============== Load & Prep ===============
print("[1/7] Load parquet ...")
train=pd.read_parquet(args.train); test=pd.read_parquet(args.test)
assert args.label in train.columns and args.seq_col in train.columns

if 0<args.sample_subset<1.0:
    print(f"Subsampling to {args.sample_subset*100:.1f}%")
    train=train.sample(frac=args.sample_subset, random_state=args.seed).reset_index(drop=True)

must_drop={args.label,args.seq_col,args.id_col}
base_cols=[c for c in train.columns if c not in must_drop]

# ---- 타깃 자동 선택 + 범주형/연속형 분리 ----
fixed_cats = [c for c in ["gender","age_group","inventory_id","day_of_week","hour"] if c in base_cols]
all_lfeats = [c for c in base_cols if c.startswith("l_feat_")]

preferred_targets = ["ad_id","creative_id","l_feat_14"]
present_targets = [c for c in preferred_targets if c in base_cols]
target_name = present_targets[0] if len(present_targets)>0 else None
if target_name is None and "l_feat_14" in all_lfeats:
    target_name = "l_feat_14"

CAT_CARD_MAX = 200_000
cand_cats = fixed_cats.copy()

# 임시 split로 카디널리티 대략 파악
tr_tmp, va_tmp = train_test_split(train, test_size=0.15, random_state=args.seed, stratify=train[args.label])
for c in all_lfeats:
    try:
        nunq = max(tr_tmp[c].nunique(dropna=True), va_tmp[c].nunique(dropna=True))
    except Exception:
        nunq = pd.concat([tr_tmp[c], va_tmp[c]], axis=0).nunique(dropna=True)
    if nunq <= CAT_CARD_MAX:
        cand_cats.append(c)

if target_name is not None and target_name not in cand_cats and target_name in base_cols:
    cand_cats.append(target_name)

cont_cols=[c for c in base_cols if c not in cand_cats]
print(f"[1.1] target_name={target_name}  cont={len(cont_cols)}  cat={len(cand_cats)}")

# 최종 split
tr,va=train_test_split(train, test_size=0.15, random_state=args.seed, stratify=train[args.label]); del train; gc.collect()

# 범주형 맵 (train 기준)
cat_maps, cat_cards = {}, {}
for c in cand_cats:
    cats=pd.Categorical(tr[c])
    cat_maps[c]={v:i for i,v in enumerate(cats.categories)}
    cat_cards[c]=len(cats.categories)

# 시퀀스 해시 임베딩
PAD_ID, OOF_ID, SEQ_BASE = 0, 1, 2
seq_vocab_size = args.HASH_BUCKETS + SEQ_BASE
def seq_to_ids_hash(lst, max_len=50):
    ids=[SEQ_BASE + (int(t)%args.HASH_BUCKETS) for t in lst][-max_len:]
    if len(ids)<max_len: ids=[PAD_ID]*(max_len-len(ids))+ids
    return np.array(ids, dtype=np.int32)

# =============== Dataset (Lazy seq parsing) ===============
class CTRDataset(Dataset):
    def __init__(self, df, cont_cols, cat_cols, cat_maps, seq_col, max_seq_len, label_col=None):
        self.df=df.reset_index(drop=True)
        self.cont_cols, self.cat_cols = cont_cols, cat_cols
        self.cat_maps, self.seq_col = cat_maps, seq_col
        self.max_seq_len=max_seq_len; self.has_label=label_col is not None; self.label_col=label_col
        self.Xc=self.df[self.cont_cols].astype(np.float32).fillna(0.0).values if self.cont_cols else None
        self.Xcats={c:self.df[c].map(self.cat_maps[c]).fillna(cat_cards[c]).astype(np.int64).values for c in self.cat_cols}
        if self.has_label: self.y=self.df[self.label_col].astype(np.float32).values
    def __len__(self): return len(self.df)
    def __getitem__(self, idx):
        xc=torch.from_numpy(self.Xc[idx]) if self.Xc is not None else torch.empty(0)
        cats={c: torch.tensor(self.Xcats[c][idx],dtype=torch.long) for c in self.cat_cols}
        lst=parse_seq_string(self.df.at[idx,self.seq_col])
        seq=torch.from_numpy(seq_to_ids_hash(lst, self.max_seq_len)).long()
        if self.has_label: return xc, cats, seq, torch.tensor(self.y[idx],dtype=torch.float32)
        return xc, cats, seq

# =============== DCN-V2 (CrossNetMix) ===============
class CrossLayerMix(nn.Module):
    def __init__(self, dim, low_rank=32, num_experts=4):
        super().__init__()
        self.num_experts=num_experts
        self.U=nn.Parameter(torch.randn(num_experts, dim, low_rank)*0.01)
        self.V=nn.Parameter(torch.randn(num_experts, low_rank, dim)*0.01)
        self.C=nn.Parameter(torch.zeros(num_experts, dim))
        self.gating=nn.Linear(dim, num_experts, bias=False)
        self.bias=nn.Parameter(torch.zeros(dim))
    def forward(self, x0, x):
        gate=torch.softmax(self.gating(x), dim=-1)   # (B,E)
        Xu=torch.einsum("bd,edr->ber", x, self.U)    # (B,E,r)
        Xuv=torch.einsum("ber,erd->bed", Xu, self.V) # (B,E,d)
        Xuv = Xuv + self.C                            # (B,E,d)
        mix=torch.einsum("be,bed->bd", gate, Xuv)    # (B,d)
        out=x0*mix + x + self.bias                   # cross + residual
        return out

class CrossNetMix(nn.Module):
    def __init__(self, dim, num_layers=3, low_rank=32, num_experts=4):
        super().__init__()
        self.layers=nn.ModuleList([CrossLayerMix(dim, low_rank, num_experts) for _ in range(num_layers)])
    def forward(self, x):
        x0=x; xl=x
        for layer in self.layers:
            xl=layer(x0, xl)
        return xl

# =============== DIN Activation Unit (공통 어텐션) ===============
class DINActivationUnit(nn.Module):
    """w_i = MLP([q, k_i, q-k_i, q*k_i])"""
    def __init__(self, dim_q, dim_k, hidden=[64,32], dropout=0.0):
        super().__init__()
        in_dim = dim_q + dim_k + dim_q + dim_k
        layers=[]
        for h in hidden:
            layers += [nn.Linear(in_dim, h), nn.ReLU(), nn.Dropout(dropout)]
            in_dim=h
        layers += [nn.Linear(in_dim, 1)]
        self.net=nn.Sequential(*layers)
    def forward(self, q, K):
        B,L,Dk = K.shape
        q_exp = q.unsqueeze(1).expand(-1,L,-1)
        feats = [q_exp, K, q_exp-K, q_exp*K]
        z = torch.cat(feats, dim=2)
        w = self.net(z).squeeze(2)    # (B,L)
        return w

# =============== Sequence Backbones ===============
class DIENBackbone(nn.Module):
    """GRU → 은닉상태 시퀀스 H에 DIN-activation attention"""
    def __init__(self, vocab_size, emb_dim, padding_idx=0, dropout=0.2):
        super().__init__()
        self.emb = nn.Embedding(vocab_size, emb_dim, padding_idx=padding_idx)
        self.gru = nn.GRU(input_size=emb_dim, hidden_size=emb_dim, batch_first=True)
        self.att = DINActivationUnit(emb_dim, emb_dim, hidden=[64,32], dropout=dropout)
    def forward(self, seq_ids, q_vec):
        K0 = self.emb(seq_ids)                 # (B,L,D)
        H, _ = self.gru(K0)                    # (B,L,D) — interest evolution
        logits_w = self.att(q_vec, H)          # (B,L)
        maskL = (seq_ids != PAD_ID)
        # fp16-safe masking & all-PAD safe
        neg = torch.finfo(logits_w.dtype).min
        logits_w = logits_w.masked_fill(~maskL, neg)
        valid = maskL.any(dim=1, keepdim=True)
        maxv = torch.where(valid,
                           logits_w.max(dim=1, keepdim=True).values,
                           torch.zeros_like(logits_w[:, :1]))
        logits_w = torch.where(valid, logits_w - maxv, torch.zeros_like(logits_w))
        alpha = torch.where(valid, torch.softmax(logits_w, dim=1), torch.zeros_like(logits_w))
        alpha = torch.nan_to_num(alpha, nan=0.0)
        interest = torch.sum(H * alpha.unsqueeze(2), dim=1)   # (B,D)
        return interest

class PositionalEncoding(nn.Module):
    def __init__(self, d_model, max_len=1024):
        super().__init__()
        pe = torch.zeros(max_len, d_model)
        position = torch.arange(0, max_len).unsqueeze(1)
        div_term = torch.exp(torch.arange(0, d_model, 2)*(-np.log(10000.0)/d_model))
        pe[:, 0::2] = torch.sin(position*div_term)
        pe[:, 1::2] = torch.cos(position*div_term)
        self.register_buffer('pe', pe.unsqueeze(0))  # (1,max_len,d)
    def forward(self, x):
        L = x.size(1)
        return x + self.pe[:, :L, :]

class BSTBackbone(nn.Module):
    """얕은 Transformer Encoder (2~4층) + PAD 마스크, 위치임베딩"""
    def __init__(self, vocab_size, emb_dim, nhead=4, num_layers=2, dim_ff=128, dropout=0.2, padding_idx=0):
        super().__init__()
        self.emb = nn.Embedding(vocab_size, emb_dim, padding_idx=padding_idx)
        self.pos = PositionalEncoding(emb_dim, max_len=2048)
        encoder_layer = nn.TransformerEncoderLayer(d_model=emb_dim, nhead=nhead, dim_feedforward=dim_ff,
                                                   batch_first=True, dropout=dropout, activation="relu", norm_first=True)
        self.enc = nn.TransformerEncoder(encoder_layer, num_layers=num_layers)
        self.att = DINActivationUnit(emb_dim, emb_dim, hidden=[64,32], dropout=dropout)
    def forward(self, seq_ids, q_vec):
        K0 = self.emb(seq_ids)                 # (B,L,D)
        K0 = self.pos(K0)
        key_pad_mask = (seq_ids == PAD_ID)     # True for PAD (B,L)
        H = self.enc(K0, src_key_padding_mask=key_pad_mask)  # (B,L,D)
        logits_w = self.att(q_vec, H)          # (B,L)
        # fp16-safe masking & all-PAD safe
        neg = torch.finfo(logits_w.dtype).min
        logits_w = logits_w.masked_fill(key_pad_mask, neg)
        valid = (~key_pad_mask).any(dim=1, keepdim=True)
        maxv = torch.where(valid,
                           logits_w.max(dim=1, keepdim=True).values,
                           torch.zeros_like(logits_w[:, :1]))
        logits_w = torch.where(valid, logits_w - maxv, torch.zeros_like(logits_w))
        alpha = torch.where(valid, torch.softmax(logits_w, dim=1), torch.zeros_like(logits_w))
        alpha = torch.nan_to_num(alpha, nan=0.0)
        interest = torch.sum(H * alpha.unsqueeze(2), dim=1)   # (B,D)
        return interest

# =============== Model: DCN-V2 + (DIEN or BST) ===============
class DCN_SEQ_Model(nn.Module):
    def __init__(self, cont_dim, cat_cards, seq_vocab_size, target_name=None,
                 seq_emb_dim=64, seq_backbone="dien",
                 bst_cfg=None, deep_units=[512,256,128],
                 cross_layers=3, cross_low_rank=32, cross_num_experts=4, dropout=0.2):
        super().__init__()
        self.has_cont=cont_dim>0
        if self.has_cont: self.bn=nn.BatchNorm1d(cont_dim)

        # categorical embeddings
        self.cat_embs=nn.ModuleDict()
        for name, card in cat_cards.items():
            dim=emb_dim_from_card(card+1)
            self.cat_embs[name]=nn.Embedding(card+1+1, dim)  # +1 OOV
        self.cat_total_dim = sum(emb.embedding_dim for emb in self.cat_embs.values())

        # base tabular x0
        self.tab_dim = (cont_dim if self.has_cont else 0) + self.cat_total_dim

        # DCN-V2
        self.cross = CrossNetMix(self.tab_dim, num_layers=cross_layers,
                                 low_rank=cross_low_rank, num_experts=cross_num_experts)

        # Deep tower
        deep_layers=[]; in_dim=self.tab_dim
        for h in deep_units:
            deep_layers += [nn.Linear(in_dim,h), nn.ReLU(), nn.Dropout(dropout)]
            in_dim=h
        self.deep = nn.Sequential(*deep_layers)

        # Query(target) 임베딩
        self.target_name = target_name if (target_name in self.cat_embs) else None
        if self.target_name is not None:
            tdim = self.cat_embs[self.target_name].embedding_dim
        else:
            tdim = seq_emb_dim
        self.proj_t = nn.Linear(tdim, seq_emb_dim, bias=False) if tdim!=seq_emb_dim else nn.Identity()

        # Sequence backbone
        self.seq_backbone = seq_backbone
        if seq_backbone == "bst":
            cfg = bst_cfg or {"nhead":4, "num_layers":2, "dim_ff":128}
            self.seq_enc = BSTBackbone(seq_vocab_size, seq_emb_dim,
                                       nhead=cfg["nhead"], num_layers=cfg["num_layers"], dim_ff=cfg["dim_ff"],
                                       dropout=dropout, padding_idx=PAD_ID)
        else:
            self.seq_enc = DIENBackbone(seq_vocab_size, seq_emb_dim, padding_idx=PAD_ID, dropout=dropout)

        # Final head: [cross_out, deep_out, interest_vec]
        final_in = self.tab_dim + deep_units[-1] + seq_emb_dim
        self.head = nn.Sequential(
            nn.Linear(final_in, 256), nn.ReLU(), nn.Dropout(dropout),
            nn.Linear(256, 1)
        )

    def forward(self, xc, xcats, seq_ids):
        # ----- Tabular -----
        outs=[]
        if self.has_cont: outs.append(self.bn(xc))
        if len(self.cat_embs)>0:
            outs.append(torch.cat([self.cat_embs[n](xcats[n]) for n in self.cat_embs], dim=1))
        x0 = torch.cat(outs, dim=1) if len(outs)>1 else outs[0]   # (B, tab_dim)

        # DCN-V2 cross / Deep tower
        x_cross = self.cross(x0)                                  # (B, tab_dim)
        x_deep  = self.deep(x0)                                   # (B, deep_dim)

        # Query(target) vector
        if self.target_name is not None:
            q = self.cat_embs[self.target_name](xcats[self.target_name])  # (B,Dt)
            q = self.proj_t(q)                                            # (B,D)
        else:
            # fallback: 평균 시퀀스를 q로 (backbone 내부에서 attention 안정처리됨)
            # DIEN/BST 모두 seq_ids를 필요로 하므로, 임시로 임베딩 평균을 사용
            # 여기서는 0 벡터를 주고 backbone에서만 관심 요약하도록 해도 무방
            q = torch.zeros(seq_ids.size(0), self.proj_t.out_features if hasattr(self.proj_t,'out_features') else seq_ids.size(-1), device=seq_ids.device)

        # Sequence → interest vector
        interest = self.seq_enc(seq_ids, q)                        # (B,D)

        # 출력 결합
        z = torch.cat([x_cross, x_deep, interest], dim=1)
        return self.head(z).squeeze(1)

# =============== Data & Loaders ===============
ds_tr=CTRDataset(tr, cont_cols, cand_cats, cat_maps, args.seq_col, args.max_seq_len, label_col=args.label)
ds_va=CTRDataset(va, cont_cols, cand_cats, cat_maps, args.seq_col, args.max_seq_len, label_col=args.label)
ds_te=CTRDataset(test, cont_cols, cand_cats, cat_maps, args.seq_col, args.max_seq_len, label_col=None)

n_pos=float((tr[args.label]==1).sum()); n_neg=float(len(tr)-n_pos)
pos_weight=torch.tensor(max(1.0, n_neg/max(1.0,n_pos)), dtype=torch.float32, device=device)
print(f"[2/7] Train label: pos={int(n_pos)} neg={int(n_neg)}  pos_weight={pos_weight.item():.3f}")

dl_tr=DataLoader(ds_tr, batch_size=args.batch_size, shuffle=True,
                 num_workers=args.num_workers, pin_memory=args.pin_memory)
dl_va=DataLoader(ds_va, batch_size=args.batch_size, shuffle=False,
                 num_workers=args.num_workers, pin_memory=args.pin_memory)
dl_te=DataLoader(ds_te, batch_size=args.batch_size, shuffle=False,
                 num_workers=args.num_workers, pin_memory=args.pin_memory)

# =============== Train ===============
print("[3/7] Build model ...")
bst_cfg={"nhead":args.bst_nhead, "num_layers":args.bst_layers, "dim_ff":args.bst_ffn}
model=DCN_SEQ_Model(
    cont_dim=len(cont_cols), cat_cards=cat_cards, seq_vocab_size=seq_vocab_size,
    target_name=target_name, seq_emb_dim=args.seq_emb_dim, seq_backbone=args.seq_backbone,
    bst_cfg=bst_cfg, deep_units=args.deep_units,
    cross_layers=args.cross_layers, cross_low_rank=args.cross_low_rank, cross_num_experts=args.cross_num_experts,
    dropout=args.dropout
).to(device)

criterion=nn.BCEWithLogitsLoss(pos_weight=pos_weight)
optimizer=torch.optim.AdamW(model.parameters(), lr=args.lr, weight_decay=1e-4)
scheduler=torch.optim.lr_scheduler.CosineAnnealingLR(optimizer, T_max=args.epochs)
scaler=GradScaler(enabled=(device.type=="cuda"))

best_auc, best_state, wait=-1.0, None, 0
final_epoch, final_prauc=0, 0.0

print(f"[4/7] Train ... (backbone={args.seq_backbone})")
for epoch in range(1, args.epochs+1):
    model.train(); tr_loss=0.0
    for xc, cats, seq, y in dl_tr:
        xc, seq, y = xc.to(device, non_blocking=True), seq.to(device, non_blocking=True), y.to(device, non_blocking=True)
        cats_dev={k:v.to(device, non_blocking=True) for k,v in cats.items()}
        optimizer.zero_grad(set_to_none=True)
        with autocast(enabled=(device.type=="cuda")):
            logits=model(xc, cats_dev, seq)
            loss=criterion(logits, y)
        scaler.scale(loss).backward(); scaler.step(optimizer); scaler.update()
        tr_loss += loss.item()*y.size(0)
    tr_loss/=len(ds_tr); scheduler.step()

    # ----- Validation -----
    model.eval(); va_loss=0.0; ys=[]; ps=[]
    with torch.no_grad():
        for xc, cats, seq, y in dl_va:
            xc, seq = xc.to(device, non_blocking=True), seq.to(device, non_blocking=True)
            cats_dev={k:v.to(device, non_blocking=True) for k,v in cats.items()}
            with autocast(enabled=(device.type=="cuda")):
                logits=model(xc, cats_dev, seq)
                loss=criterion(logits, y.to(device))
                prob=torch.sigmoid(logits)
            p = prob.detach().cpu().numpy().astype(np.float64)
            p = np.nan_to_num(p, nan=0.0, posinf=1.0, neginf=0.0)
            va_loss += loss.item()*len(y)
            ys.append(y.numpy()); ps.append(p)
    va_loss/=len(ds_va)
    y_true=np.concatenate(ys); y_prob=np.concatenate(ps)
    auc=roc_auc_score(y_true, y_prob); prauc=average_precision_score(y_true, y_prob)
    print(f"Epoch {epoch:02d} | Train {tr_loss:.5f} | Val {va_loss:.5f} | AUC {auc:.6f} | PR-AUC {prauc:.6f}")

    if auc > best_auc + 1e-5:
        best_auc, final_prauc, final_epoch = auc, prauc, epoch
        best_state={k:v.cpu().clone() for k,v in model.state_dict().items()}; wait=0
    else:
        wait += 1
        if wait >= args.patience:
            print(f"Early stopping. Best AUC={best_auc:.6f} at epoch {final_epoch}."); break

if best_state is not None:
    model.load_state_dict(best_state)

# =============== Inference ===============
print("[5/7] Predict test ...")
model.eval(); probs=[]
with torch.no_grad():
    for xc, cats, seq in dl_te:
        xc, seq = xc.to(device, non_blocking=True), seq.to(device, non_blocking=True)
        cats_dev={k:v.to(device, non_blocking=True) for k,v in cats.items()}
        with autocast(enabled=(device.type=="cuda")):
            logits=model(xc, cats_dev, seq)
            prob=torch.sigmoid(logits)
        p = prob.detach().cpu().numpy().astype(np.float64)
        p = np.nan_to_num(p, nan=0.0, posinf=1.0, neginf=0.0)
        probs.append(p)
probs=np.concatenate(probs)

# =============== Save ===============
print("[6/7] Save outputs ...]")
sub_path=f"./dcnv2_{args.seq_backbone}_submit.csv"; meta_path=f"./dcnv2_{args.seq_backbone}_meta.json"
pd.DataFrame({args.id_col: test[args.id_col].values, "clicked": probs}).to_csv(sub_path, index=False)
meta={
    "columns":{"continuous":cont_cols, "categorical":cand_cats},
    "seq_vocab":{"type":"hash","buckets":args.HASH_BUCKETS,"pad_id":0,"oov_id":1},
    "hyperparameters":{
        "sample_subset":args.sample_subset,"max_seq_len":args.max_seq_len,
        "seq_emb_dim":args.seq_emb_dim,"dropout":args.dropout,
        "epochs":args.epochs,"batch_size":args.batch_size,"lr":args.lr,
        "cross_layers":args.cross_layers,"cross_low_rank":args.cross_low_rank,"cross_num_experts":args.cross_num_experts,
        "deep_units":args.deep_units, "seq_backbone":args.seq_backbone,
        "bst_layers":args.bst_layers,"bst_nhead":args.bst_nhead,"bst_ffn":args.bst_ffn
    },
    "performance":{"best_epoch":int(final_epoch),"AUC":float(best_auc),"PR_AUC":float(final_prauc)},
    "target_name": target_name
}
with open(meta_path,"w",encoding="utf-8") as f: json.dump(meta,f,ensure_ascii=False,indent=2)
print(f"[7/7] Done. \n - {sub_path}\n - {meta_path}")


Using CUDA
[1/7] Load parquet ...
[1.1] target_name=l_feat_14  cont=85  cat=32
[2/7] Train label: pos=173552 neg=8925000  pos_weight=51.426
[3/7] Build model ...
[4/7] Train ... (backbone=dien)
Epoch 01 | Train 1.20800 | Val nan | AUC 0.731169 | PR-AUC 0.065352
Epoch 02 | Train 1.18786 | Val nan | AUC 0.738230 | PR-AUC 0.074571
Epoch 03 | Train 1.16517 | Val nan | AUC 0.741283 | PR-AUC 0.079196
[5/7] Predict test ...
[6/7] Save outputs ...]
[7/7] Done. 
 - ./dcnv2_dien_submit.csv
 - ./dcnv2_dien_meta.json


In [1]:
# -*- coding: utf-8 -*-
"""
CTR Baseline — DCN-V2 + (DIEN / BST / AUTO-BST)  [Notebook, OOM-safe, NaN-safe]
- 기존 안정화(AMP 마스크, 전부 PAD, NaN 정리) 유지
- 시퀀스 백본:
    * "dien": GRU + DIN activation attention
    * "bst":  Transformer Encoder (얕게)
    * "auto_bst": 소샘플/1epoch로 BST 후보 자동 탐색 → 최적 config로 본학습
"""

import os, re, json, random, gc, warnings, math
warnings.filterwarnings("ignore")
import numpy as np, pandas as pd
from sklearn.metrics import roc_auc_score, average_precision_score
from sklearn.model_selection import train_test_split
import torch, torch.nn as nn, torch.nn.functional as F
from torch.utils.data import Dataset, DataLoader
from torch.cuda.amp import autocast, GradScaler

# =============== Utils ===============
def set_seed(seed=42):
    random.seed(seed); os.environ["PYTHONHASHSEED"]=str(seed)
    np.random.seed(seed); torch.manual_seed(seed); torch.cuda.manual_seed_all(seed)
    torch.backends.cudnn.deterministic=True; torch.backends.cudnn.benchmark=False

def infer_device(arg="auto"):
    if arg=="cpu": return torch.device("cpu")
    if arg=="cuda" or (arg=="auto" and torch.cuda.is_available()):
        print("Using CUDA"); return torch.device("cuda")
    print("Using CPU"); return torch.device("cpu")

def parse_seq_string(s: str):
    if s is None: return []
    s = str(s).strip()
    if not s or s.lower()=="nan": return []
    if s.startswith("[") and s.endswith("]"):
        s = s[1:-1]; toks = [t.strip().strip("'\"") for t in s.split(",")]
    else:
        s = re.sub(r"[^\d]+", ",", s); toks = [t for t in s.split(",") if t]
    out=[]
    for t in toks:
        try: out.append(int(t))
        except Exception:
            try: out.append(int(float(t)))
            except Exception: pass
    return out

def emb_dim_from_card(card, cap=64):
    return int(min(cap, max(8, round(1.6*(card**0.25)))))

# =============== Config ===============
class Args:
    train="./train.parquet"; test="./test.parquet"
    label="clicked"; seq_col="seq"; id_col="ID"

    sample_subset=1.0
    max_seq_len=50
    HASH_BUCKETS=262_144
    seq_emb_dim=64
    dropout=0.2

    # 시퀀스 백본: "dien" | "bst" | "auto_bst"
    seq_backbone="auto_bst"
    # BST 기본값(자동탐색 실패/스킵 시 사용)
    bst_layers=2
    bst_nhead=4
    bst_ffn=128

    # AUTO-BST 탐색 설정
    autotune_frac=0.02        # 2% 소샘플
    autotune_epochs=1         # 후보당 1 epoch
    autotune_candidates=(     # 무거운→가벼운 순서로 두면 OOM 테스트가 빨라짐
        {"layers":3,"nhead":8,"ffn":256},
        {"layers":3,"nhead":4,"ffn":256},
        {"layers":2,"nhead":4,"ffn":256},
        {"layers":2,"nhead":4,"ffn":128},
    )

    # DCN-V2 (CrossNetMix)
    cross_layers=3
    cross_low_rank=32
    cross_num_experts=4

    # Deep tower
    deep_units=[512,256,128]

    # Train
    epochs=3; batch_size=512; lr=1e-3; patience=2
    seed=42; device="auto"; num_workers=0; pin_memory=False
args=Args()
set_seed(args.seed); device=infer_device(args.device)

# =============== (옵션) 더미 데이터 생성 (파일 없을 때만) ===============
def create_dummy(num_rows, is_test=False):
    rng=np.random.default_rng(0)
    df=pd.DataFrame({
        "cont_feat1": rng.normal(size=num_rows),
        "cont_feat2": rng.random(num_rows)*10,
        "gender": rng.choice(["M","F","U"], num_rows, p=[0.45,0.45,0.1]),
        "age_group": rng.choice(["10s","20s","30s","40s+"], num_rows),
        "inventory_id": rng.integers(500, 520, num_rows),
        "day_of_week": rng.integers(0,7,num_rows),
        "hour": rng.integers(0,24,num_rows),
        "l_feat_14": rng.integers(1000,1050,num_rows),
    })
    seqs=[]
    for _ in range(num_rows):
        fmt=rng.choice(["pipe","comma","json","empty","single"])
        L=int(rng.integers(1,args.max_seq_len+10))
        items=[str(int(rng.integers(1000,1500))) for _ in range(L)]
        if fmt=="pipe": seqs.append("|".join(items))
        elif fmt=="comma": seqs.append(",".join(items))
        elif fmt=="json": seqs.append(f"[{','.join(items)}]")
        elif fmt=="empty": seqs.append(np.nan)
        else: seqs.append(items[0])
    df[args.seq_col]=seqs
    if is_test: df[args.id_col]=np.arange(num_rows)
    else: df[args.label]=rng.integers(0,2,num_rows)
    return df

if not os.path.exists(args.train):
    print("train.parquet 없음 → 더미 20k 생성"); create_dummy(20_000,False).to_parquet(args.train)
if not os.path.exists(args.test):
    print("test.parquet 없음 → 더미 5k 생성"); create_dummy(5_000,True).to_parquet(args.test)

# =============== Load & Prep ===============
print("[1/9] Load parquet ...")
train=pd.read_parquet(args.train); test=pd.read_parquet(args.test)
assert args.label in train.columns and args.seq_col in train.columns

if 0<args.sample_subset<1.0:
    print(f"Subsampling to {args.sample_subset*100:.1f}%")
    train=train.sample(frac=args.sample_subset, random_state=args.seed).reset_index(drop=True)

must_drop={args.label,args.seq_col,args.id_col}
base_cols=[c for c in train.columns if c not in must_drop]

# ---- 타깃 자동 선택 + 범주형/연속형 분리 ----
fixed_cats = [c for c in ["gender","age_group","inventory_id","day_of_week","hour"] if c in base_cols]
all_lfeats = [c for c in base_cols if c.startswith("l_feat_")]

preferred_targets = ["ad_id","creative_id","l_feat_14"]
present_targets = [c for c in preferred_targets if c in base_cols]
target_name = present_targets[0] if len(present_targets)>0 else None
if target_name is None and "l_feat_14" in all_lfeats:
    target_name = "l_feat_14"

CAT_CARD_MAX = 200_000
cand_cats = fixed_cats.copy()

# 임시 split로 카디널리티 대략 파악
tr_tmp, va_tmp = train_test_split(train, test_size=0.15, random_state=args.seed, stratify=train[args.label])
for c in all_lfeats:
    try:
        nunq = max(tr_tmp[c].nunique(dropna=True), va_tmp[c].nunique(dropna=True))
    except Exception:
        nunq = pd.concat([tr_tmp[c], va_tmp[c]], axis=0).nunique(dropna=True)
    if nunq <= CAT_CARD_MAX:
        cand_cats.append(c)

if target_name is not None and target_name not in cand_cats and target_name in base_cols:
    cand_cats.append(target_name)

cont_cols=[c for c in base_cols if c not in cand_cats]
print(f"[1.1] target_name={target_name}  cont={len(cont_cols)}  cat={len(cand_cats)}")

# 최종 split
tr,va=train_test_split(train, test_size=0.15, random_state=args.seed, stratify=train[args.label]); del train; gc.collect()

# 범주형 맵 (train 기준)
cat_maps, cat_cards = {}, {}
for c in cand_cats:
    cats=pd.Categorical(tr[c])
    cat_maps[c]={v:i for i,v in enumerate(cats.categories)}
    cat_cards[c]=len(cats.categories)

# 시퀀스 해시 임베딩
PAD_ID, OOF_ID, SEQ_BASE = 0, 1, 2
seq_vocab_size = args.HASH_BUCKETS + SEQ_BASE
def seq_to_ids_hash(lst, max_len=50):
    ids=[SEQ_BASE + (int(t)%args.HASH_BUCKETS) for t in lst][-max_len:]
    if len(ids)<max_len: ids=[PAD_ID]*(max_len-len(ids))+ids
    return np.array(ids, dtype=np.int32)

# =============== Dataset (Lazy seq parsing) ===============
class CTRDataset(Dataset):
    def __init__(self, df, cont_cols, cat_cols, cat_maps, seq_col, max_seq_len, label_col=None):
        self.df=df.reset_index(drop=True)
        self.cont_cols, self.cat_cols = cont_cols, cat_cols
        self.cat_maps, self.seq_col = cat_maps, seq_col
        self.max_seq_len=max_seq_len; self.has_label=label_col is not None; self.label_col=label_col
        self.Xc=self.df[self.cont_cols].astype(np.float32).fillna(0.0).values if self.cont_cols else None
        self.Xcats={c:self.df[c].map(self.cat_maps[c]).fillna(cat_cards[c]).astype(np.int64).values for c in self.cat_cols}
        if self.has_label: self.y=self.df[self.label_col].astype(np.float32).values
    def __len__(self): return len(self.df)
    def __getitem__(self, idx):
        xc=torch.from_numpy(self.Xc[idx]) if self.Xc is not None else torch.empty(0)
        cats={c: torch.tensor(self.Xcats[c][idx],dtype=torch.long) for c in self.cat_cols}
        lst=parse_seq_string(self.df.at[idx,self.seq_col])
        seq=torch.from_numpy(seq_to_ids_hash(lst, self.max_seq_len)).long()
        if self.has_label: return xc, cats, seq, torch.tensor(self.y[idx],dtype=torch.float32)
        return xc, cats, seq

# =============== DCN-V2 (CrossNetMix) ===============
class CrossLayerMix(nn.Module):
    def __init__(self, dim, low_rank=32, num_experts=4):
        super().__init__()
        self.num_experts=num_experts
        self.U=nn.Parameter(torch.randn(num_experts, dim, low_rank)*0.01)
        self.V=nn.Parameter(torch.randn(num_experts, low_rank, dim)*0.01)
        self.C=nn.Parameter(torch.zeros(num_experts, dim))
        self.gating=nn.Linear(dim, num_experts, bias=False)
        self.bias=nn.Parameter(torch.zeros(dim))
    def forward(self, x0, x):
        gate=torch.softmax(self.gating(x), dim=-1)   # (B,E)
        Xu=torch.einsum("bd,edr->ber", x, self.U)    # (B,E,r)
        Xuv=torch.einsum("ber,erd->bed", Xu, self.V) # (B,E,d)
        Xuv = Xuv + self.C                            # (B,E,d)
        mix=torch.einsum("be,bed->bd", gate, Xuv)    # (B,d)
        out=x0*mix + x + self.bias                   # cross + residual
        return out

class CrossNetMix(nn.Module):
    def __init__(self, dim, num_layers=3, low_rank=32, num_experts=4):
        super().__init__()
        self.layers=nn.ModuleList([CrossLayerMix(dim, low_rank, num_experts) for _ in range(num_layers)])
    def forward(self, x):
        x0=x; xl=x
        for layer in self.layers:
            xl=layer(x0, xl)
        return xl

# =============== DIN Activation Unit (공통 어텐션) ===============
class DINActivationUnit(nn.Module):
    """w_i = MLP([q, k_i, q-k_i, q*k_i])"""
    def __init__(self, dim_q, dim_k, hidden=[64,32], dropout=0.0):
        super().__init__()
        in_dim = dim_q + dim_k + dim_q + dim_k
        layers=[]
        for h in hidden:
            layers += [nn.Linear(in_dim, h), nn.ReLU(), nn.Dropout(dropout)]
            in_dim=h
        layers += [nn.Linear(in_dim, 1)]
        self.net=nn.Sequential(*layers)
    def forward(self, q, K):
        B,L,Dk = K.shape
        q_exp = q.unsqueeze(1).expand(-1,L,-1)
        feats = [q_exp, K, q_exp-K, q_exp*K]
        z = torch.cat(feats, dim=2)
        w = self.net(z).squeeze(2)    # (B,L)
        return w

# =============== Sequence Backbones ===============
class DIENBackbone(nn.Module):
    """GRU → 은닉상태 시퀀스 H에 DIN-activation attention"""
    def __init__(self, vocab_size, emb_dim, padding_idx=0, dropout=0.2):
        super().__init__()
        self.emb = nn.Embedding(vocab_size, emb_dim, padding_idx=padding_idx)
        self.gru = nn.GRU(input_size=emb_dim, hidden_size=emb_dim, batch_first=True)
        self.att = DINActivationUnit(emb_dim, emb_dim, hidden=[64,32], dropout=dropout)
    def forward(self, seq_ids, q_vec):
        K0 = self.emb(seq_ids)
        H, _ = self.gru(K0)
        logits_w = self.att(q_vec, H)
        maskL = (seq_ids != PAD_ID)
        neg = torch.finfo(logits_w.dtype).min
        logits_w = logits_w.masked_fill(~maskL, neg)
        valid = maskL.any(dim=1, keepdim=True)
        maxv = torch.where(valid, logits_w.max(dim=1, keepdim=True).values,
                           torch.zeros_like(logits_w[:, :1]))
        logits_w = torch.where(valid, logits_w - maxv, torch.zeros_like(logits_w))
        alpha = torch.where(valid, torch.softmax(logits_w, dim=1), torch.zeros_like(logits_w))
        alpha = torch.nan_to_num(alpha, nan=0.0)
        interest = torch.sum(H * alpha.unsqueeze(2), dim=1)
        return interest

class PositionalEncoding(nn.Module):
    def __init__(self, d_model, max_len=2048):
        super().__init__()
        pe = torch.zeros(max_len, d_model)
        position = torch.arange(0, max_len).unsqueeze(1)
        div_term = torch.exp(torch.arange(0, d_model, 2)*(-math.log(10000.0)/d_model))
        pe[:, 0::2] = torch.sin(position*div_term)
        pe[:, 1::2] = torch.cos(position*div_term)
        self.register_buffer('pe', pe.unsqueeze(0))  # (1,max_len,d)
    def forward(self, x):
        L = x.size(1)
        return x + self.pe[:, :L, :]

class BSTBackbone(nn.Module):
    """얕은 Transformer Encoder + PAD 마스크, 위치임베딩"""
    def __init__(self, vocab_size, emb_dim, nhead=4, num_layers=2, dim_ff=128, dropout=0.2, padding_idx=0):
        super().__init__()
        self.emb = nn.Embedding(vocab_size, emb_dim, padding_idx=padding_idx)
        self.pos = PositionalEncoding(emb_dim, max_len=2048)
        enc_layer = nn.TransformerEncoderLayer(d_model=emb_dim, nhead=nhead, dim_feedforward=dim_ff,
                                               batch_first=True, dropout=dropout, activation="relu", norm_first=True)
        self.enc = nn.TransformerEncoder(enc_layer, num_layers=num_layers)
        self.att = DINActivationUnit(emb_dim, emb_dim, hidden=[64,32], dropout=dropout)
        self.padding_idx = padding_idx
    def forward(self, seq_ids, q_vec):
        K0 = self.emb(seq_ids)
        K0 = self.pos(K0)
        key_pad_mask = (seq_ids == self.padding_idx)     # (B,L) True for PAD
        H = self.enc(K0, src_key_padding_mask=key_pad_mask)
        logits_w = self.att(q_vec, H)
        neg = torch.finfo(logits_w.dtype).min
        logits_w = logits_w.masked_fill(key_pad_mask, neg)
        valid = (~key_pad_mask).any(dim=1, keepdim=True)
        maxv = torch.where(valid,
                           logits_w.max(dim=1, keepdim=True).values,
                           torch.zeros_like(logits_w[:, :1]))
        logits_w = torch.where(valid, logits_w - maxv, torch.zeros_like(logits_w))
        alpha = torch.where(valid, torch.softmax(logits_w, dim=1), torch.zeros_like(logits_w))
        alpha = torch.nan_to_num(alpha, nan=0.0)
        interest = torch.sum(H * alpha.unsqueeze(2), dim=1)
        return interest

# =============== Model: DCN-V2 + (DIEN / BST) ===============
class DCN_SEQ_Model(nn.Module):
    def __init__(self, cont_dim, cat_cards, seq_vocab_size, target_name=None,
                 seq_emb_dim=64, seq_backbone="dien",
                 bst_cfg=None, deep_units=[512,256,128],
                 cross_layers=3, cross_low_rank=32, cross_num_experts=4, dropout=0.2):
        super().__init__()
        self.has_cont=cont_dim>0
        if self.has_cont: self.bn=nn.BatchNorm1d(cont_dim)

        # categorical embeddings
        self.cat_embs=nn.ModuleDict()
        for name, card in cat_cards.items():
            dim=emb_dim_from_card(card+1)
            self.cat_embs[name]=nn.Embedding(card+1+1, dim)  # +1 OOV
        self.cat_total_dim = sum(emb.embedding_dim for emb in self.cat_embs.values())

        # base tabular x0
        self.tab_dim = (cont_dim if self.has_cont else 0) + self.cat_total_dim

        # DCN-V2
        self.cross = CrossNetMix(self.tab_dim, num_layers=cross_layers,
                                 low_rank=cross_low_rank, num_experts=cross_num_experts)

        # Deep tower
        deep_layers=[]; in_dim=self.tab_dim
        for h in deep_units:
            deep_layers += [nn.Linear(in_dim,h), nn.ReLU(), nn.Dropout(dropout)]
            in_dim=h
        self.deep = nn.Sequential(*deep_layers)

        # Query(target) 임베딩
        self.target_name = target_name if (target_name in self.cat_embs) else None
        if self.target_name is not None:
            tdim = self.cat_embs[self.target_name].embedding_dim
        else:
            tdim = seq_emb_dim
        self.proj_t = nn.Linear(tdim, seq_emb_dim, bias=False) if tdim!=seq_emb_dim else nn.Identity()

        # Sequence backbone
        self.seq_backbone = seq_backbone
        if seq_backbone == "bst":
            cfg = bst_cfg or {"nhead":4, "num_layers":2, "dim_ff":128}
            self.seq_enc = BSTBackbone(seq_vocab_size, seq_emb_dim,
                                       nhead=cfg["nhead"], num_layers=cfg["num_layers"], dim_ff=cfg["dim_ff"],
                                       dropout=dropout, padding_idx=PAD_ID)
        else:
            self.seq_enc = DIENBackbone(seq_vocab_size, seq_emb_dim, padding_idx=PAD_ID, dropout=dropout)

        # Final head: [cross_out, deep_out, interest_vec]
        final_in = self.tab_dim + deep_units[-1] + seq_emb_dim
        self.head = nn.Sequential(
            nn.Linear(final_in, 256), nn.ReLU(), nn.Dropout(dropout),
            nn.Linear(256, 1)
        )

    def forward(self, xc, xcats, seq_ids):
        # ----- Tabular -----
        outs=[]
        if self.has_cont: outs.append(self.bn(xc))
        if len(self.cat_embs)>0:
            outs.append(torch.cat([self.cat_embs[n](xcats[n]) for n in self.cat_embs], dim=1))
        x0 = torch.cat(outs, dim=1) if len(outs)>1 else outs[0]   # (B, tab_dim)

        # DCN-V2 cross / Deep tower
        x_cross = self.cross(x0)
        x_deep  = self.deep(x0)

        # Query(target) vector
        if self.target_name is not None:
            q = self.cat_embs[self.target_name](xcats[self.target_name])  # (B,Dt)
            q = self.proj_t(q)                                            # (B,D)
        else:
            q = torch.zeros(seq_ids.size(0), self.proj_t.out_features if hasattr(self.proj_t,'out_features') else seq_ids.size(-1), device=seq_ids.device)

        # Sequence → interest vector
        interest = self.seq_enc(seq_ids, q)                        # (B,D)

        # 출력 결합
        z = torch.cat([x_cross, x_deep, interest], dim=1)
        return self.head(z).squeeze(1)

# =============== Data & Loaders ===============
def make_loaders(tr_df, va_df, batch_size, num_workers, pin_memory):
    ds_tr=CTRDataset(tr_df, cont_cols, cand_cats, cat_maps, args.seq_col, args.max_seq_len, label_col=args.label)
    ds_va=CTRDataset(va_df, cont_cols, cand_cats, cat_maps, args.seq_col, args.max_seq_len, label_col=args.label)
    dl_tr=DataLoader(ds_tr, batch_size=batch_size, shuffle=True, num_workers=num_workers, pin_memory=pin_memory)
    dl_va=DataLoader(ds_va, batch_size=batch_size, shuffle=False, num_workers=num_workers, pin_memory=pin_memory)
    return ds_tr, ds_va, dl_tr, dl_va

# 원본 전체용
ds_te=CTRDataset(test, cont_cols, cand_cats, cat_maps, args.seq_col, args.max_seq_len, label_col=None)

# =============== AUTO-BST: 소샘플 후보 탐색 ===============
def try_bst_candidate(cfg, tr_small, va_small, pos_weight):
    """주어진 cfg로 1 epoch 학습 후 Val AUC 반환. OOM 시 None."""
    torch.cuda.empty_cache() if device.type=="cuda" else None
    # 로더 (가볍게)
    bs = min(args.batch_size, 1024)
    ds_tr, ds_va, dl_tr, dl_va = make_loaders(tr_small, va_small, bs, args.num_workers, args.pin_memory)

    model=DCN_SEQ_Model(
        cont_dim=len(cont_cols), cat_cards=cat_cards, seq_vocab_size=seq_vocab_size,
        target_name=target_name, seq_emb_dim=args.seq_emb_dim, seq_backbone="bst",
        bst_cfg={"nhead":cfg["nhead"], "num_layers":cfg["layers"], "dim_ff":cfg["ffn"]},
        deep_units=args.deep_units,
        cross_layers=args.cross_layers, cross_low_rank=args.cross_low_rank, cross_num_experts=args.cross_num_experts,
        dropout=args.dropout
    ).to(device)

    criterion=nn.BCEWithLogitsLoss(pos_weight=pos_weight)
    optimizer=torch.optim.AdamW(model.parameters(), lr=args.lr, weight_decay=1e-4)
    scaler=GradScaler(enabled=(device.type=="cuda"))
    model.train()

    try:
        # 1 epoch만
        for xc, cats, seq, y in dl_tr:
            xc, seq, y = xc.to(device, non_blocking=True), seq.to(device, non_blocking=True), y.to(device, non_blocking=True)
            cats_dev={k:v.to(device, non_blocking=True) for k,v in cats.items()}
            optimizer.zero_grad(set_to_none=True)
            with autocast(enabled=(device.type=="cuda")):
                logits=model(xc, cats_dev, seq)
                loss=criterion(logits, y)
            scaler.scale(loss).backward(); scaler.step(optimizer); scaler.update()
        # Val
        model.eval(); ys=[]; ps=[]
        with torch.no_grad():
            for xc, cats, seq, y in dl_va:
                xc, seq = xc.to(device, non_blocking=True), seq.to(device, non_blocking=True)
                cats_dev={k:v.to(device, non_blocking=True) for k,v in cats.items()}
                with autocast(enabled=(device.type=="cuda")):
                    prob=torch.sigmoid(model(xc, cats_dev, seq))
                p = prob.detach().cpu().numpy().astype(np.float64)
                p = np.nan_to_num(p, nan=0.0, posinf=1.0, neginf=0.0)
                ys.append(y.numpy()); ps.append(p)
        y_true=np.concatenate(ys); y_prob=np.concatenate(ps)
        auc=roc_auc_score(y_true, y_prob)
        return auc
    except RuntimeError as e:
        if "out of memory" in str(e).lower():
            print(f"  OOM → skip cfg={cfg}")
            if device.type=="cuda":
                torch.cuda.empty_cache()
            return None
        else:
            raise e

def autotune_bst(tr, va):
    print("[2/9] AUTO-BST: build 2% mini-splits for search ...")
    frac = min(max(args.autotune_frac, 0.01), 0.1)  # 1~10% 사이로 클램프
    tr_small = tr.sample(frac=frac, random_state=args.seed)
    va_small = va.sample(frac=min(frac*1.5, 0.15), random_state=args.seed)

    # 클래스 가중치(소샘플 기준)
    n_pos=float((tr_small[args.label]==1).sum()); n_neg=float(len(tr_small)-n_pos)
    pos_weight=torch.tensor(max(1.0, n_neg/max(1.0,n_pos)), dtype=torch.float32, device=device)

    print(f"[3/9] AUTO-BST: candidates={list(args.autotune_candidates)}")
    best_auc=-1.0; best_cfg=None
    for cfg in args.autotune_candidates:
        print(f"  try cfg={cfg} (epochs={args.autotune_epochs})")
        auc = try_bst_candidate(cfg, tr_small, va_small, pos_weight)
        if auc is None:  # OOM
            continue
        print(f"   -> val AUC={auc:.6f}")
        if auc>best_auc:
            best_auc=auc; best_cfg=cfg
    return best_cfg, best_auc

# =============== 메인 학습 준비 ===============
ds_tr_full, ds_va_full, dl_tr_full, dl_va_full = make_loaders(tr, va, args.batch_size, args.num_workers, args.pin_memory)
n_pos=float((tr[args.label]==1).sum()); n_neg=float(len(tr)-n_pos)
pos_weight=torch.tensor(max(1.0, n_neg/max(1.0,n_pos)), dtype=torch.float32, device=device)
print(f"[4/9] Train label: pos={int(n_pos)} neg={int(n_neg)}  pos_weight={pos_weight.item():.3f}")

# AUTO-BST 수행 (필요 시)
chosen_seq_backbone = args.seq_backbone
bst_cfg={"nhead":args.bst_nhead, "num_layers":args.bst_layers, "dim_ff":args.bst_ffn}
auto_result=None
if args.seq_backbone == "auto_bst":
    best_cfg, best_auc = autotune_bst(tr, va)
    if best_cfg is None:
        print("AUTO-BST: all candidates OOM/failed → fallback to default BST")
        chosen_seq_backbone="bst"
    else:
        print(f"AUTO-BST: best={best_cfg} (val AUC={best_auc:.6f})")
        chosen_seq_backbone="bst"
        bst_cfg={"nhead":best_cfg["nhead"], "num_layers":best_cfg["layers"], "dim_ff":best_cfg["ffn"]}
        auto_result={"best_cfg":best_cfg, "best_auc":float(best_auc)}

# =============== Model build (final) ===============
print(f"[5/9] Build final model ... backbone={chosen_seq_backbone}, bst_cfg={bst_cfg}")
model=DCN_SEQ_Model(
    cont_dim=len(cont_cols), cat_cards=cat_cards, seq_vocab_size=seq_vocab_size,
    target_name=target_name, seq_emb_dim=args.seq_emb_dim, seq_backbone=chosen_seq_backbone,
    bst_cfg=bst_cfg, deep_units=args.deep_units,
    cross_layers=args.cross_layers, cross_low_rank=args.cross_low_rank, cross_num_experts=args.cross_num_experts,
    dropout=args.dropout
).to(device)

criterion=nn.BCEWithLogitsLoss(pos_weight=pos_weight)
optimizer=torch.optim.AdamW(model.parameters(), lr=args.lr, weight_decay=1e-4)
scheduler=torch.optim.lr_scheduler.CosineAnnealingLR(optimizer, T_max=args.epochs)
scaler=GradScaler(enabled=(device.type=="cuda"))

best_auc, best_state, wait=-1.0, None, 0
final_epoch, final_prauc=0, 0.0

# =============== Train ===============
print("[6/9] Train ...")
for epoch in range(1, args.epochs+1):
    model.train(); tr_loss=0.0
    for xc, cats, seq, y in dl_tr_full:
        xc, seq, y = xc.to(device, non_blocking=True), seq.to(device, non_blocking=True), y.to(device, non_blocking=True)
        cats_dev={k:v.to(device, non_blocking=True) for k,v in cats.items()}
        optimizer.zero_grad(set_to_none=True)
        with autocast(enabled=(device.type=="cuda")):
            logits=model(xc, cats_dev, seq)
            loss=criterion(logits, y)
        scaler.scale(loss).backward(); scaler.step(optimizer); scaler.update()
        tr_loss += loss.item()*y.size(0)
    tr_loss/=len(ds_tr_full); scheduler.step()

    # Val
    model.eval(); va_loss=0.0; ys=[]; ps=[]
    with torch.no_grad():
        for xc, cats, seq, y in dl_va_full:
            xc, seq = xc.to(device, non_blocking=True), seq.to(device, non_blocking=True)
            cats_dev={k:v.to(device, non_blocking=True) for k,v in cats.items()}
            with autocast(enabled=(device.type=="cuda")):
                prob=torch.sigmoid(model(xc, cats_dev, seq))
                loss=criterion(prob.logit() if hasattr(prob,'logit') else torch.logit(prob.clamp(1e-6,1-1e-6)), y.to(device))  # 안전
            p = prob.detach().cpu().numpy().astype(np.float64)
            p = np.nan_to_num(p, nan=0.0, posinf=1.0, neginf=0.0)
            va_loss += loss.item()*len(y)
            ys.append(y.numpy()); ps.append(p)
    va_loss/=len(ds_va_full)
    y_true=np.concatenate(ys); y_prob=np.concatenate(ps)
    auc=roc_auc_score(y_true, y_prob); prauc=average_precision_score(y_true, y_prob)
    print(f"Epoch {epoch:02d} | Train {tr_loss:.5f} | Val {va_loss:.5f} | AUC {auc:.6f} | PR-AUC {prauc:.6f}")

    if auc > best_auc + 1e-5:
        best_auc, final_prauc, final_epoch = auc, prauc, epoch
        best_state={k:v.cpu().clone() for k,v in model.state_dict().items()}; wait=0
    else:
        wait += 1
        if wait >= args.patience:
            print(f"Early stopping. Best AUC={best_auc:.6f} at epoch {final_epoch}."); break

if best_state is not None:
    model.load_state_dict(best_state)

# =============== Inference ===============
print("[7/9] Predict test ...")
dl_te=DataLoader(ds_te, batch_size=args.batch_size, shuffle=False, num_workers=args.num_workers, pin_memory=args.pin_memory)
model.eval(); probs=[]
with torch.no_grad():
    for xc, cats, seq in dl_te:
        xc, seq = xc.to(device, non_blocking=True), seq.to(device, non_blocking=True)
        cats_dev={k:v.to(device, non_blocking=True) for k,v in cats.items()}
        with autocast(enabled=(device.type=="cuda")):
            prob=torch.sigmoid(model(xc, cats_dev, seq))
        p = prob.detach().cpu().numpy().astype(np.float64)
        p = np.nan_to_num(p, nan=0.0, posinf=1.0, neginf=0.0)
        probs.append(p)
probs=np.concatenate(probs)

# =============== Save ===============
print("[8/9] Save outputs ...")
suffix = "auto_bst" if args.seq_backbone=="auto_bst" else args.seq_backbone
sub_path=f"./dcnv2_{suffix}_submit.csv"; meta_path=f"./dcnv2_{suffix}_meta.json"
pd.DataFrame({args.id_col: test[args.id_col].values, "clicked": probs}).to_csv(sub_path, index=False)
meta={
    "columns":{"continuous":cont_cols, "categorical":cand_cats},
    "seq_vocab":{"type":"hash","buckets":args.HASH_BUCKETS,"pad_id":0,"oov_id":1},
    "hyperparameters":{
        "sample_subset":args.sample_subset,"max_seq_len":args.max_seq_len,
        "seq_emb_dim":args.seq_emb_dim,"dropout":args.dropout,
        "epochs":args.epochs,"batch_size":args.batch_size,"lr":args.lr,
        "cross_layers":args.cross_layers,"cross_low_rank":args.cross_low_rank,"cross_num_experts":args.cross_num_experts,
        "deep_units":args.deep_units, "seq_backbone":suffix,
        "bst_layers":bst_cfg["num_layers"],"bst_nhead":bst_cfg["nhead"],"bst_ffn":bst_cfg["dim_ff"],
        "autotune_frac":args.autotune_frac,"autotune_epochs":args.autotune_epochs,
        "autotune_candidates":list(args.autotune_candidates)
    },
    "performance":{"best_epoch":int(final_epoch),"AUC":float(best_auc),"PR_AUC":float(final_prauc)},
    "target_name": target_name,
    "autotune_result": auto_result
}
with open(meta_path,"w",encoding="utf-8") as f: json.dump(meta,f,ensure_ascii=False,indent=2)
print(f"[9/9] Done. \n - {sub_path}\n - {meta_path}")


Using CUDA
[1/9] Load parquet ...
[1.1] target_name=l_feat_14  cont=85  cat=32
[4/9] Train label: pos=173552 neg=8925000  pos_weight=51.426
[2/9] AUTO-BST: build 2% mini-splits for search ...
[3/9] AUTO-BST: candidates=[{'layers': 3, 'nhead': 8, 'ffn': 256}, {'layers': 3, 'nhead': 4, 'ffn': 256}, {'layers': 2, 'nhead': 4, 'ffn': 256}, {'layers': 2, 'nhead': 4, 'ffn': 128}]
  try cfg={'layers': 3, 'nhead': 8, 'ffn': 256} (epochs=1)
   -> val AUC=0.713384
  try cfg={'layers': 3, 'nhead': 4, 'ffn': 256} (epochs=1)
   -> val AUC=0.702857
  try cfg={'layers': 2, 'nhead': 4, 'ffn': 256} (epochs=1)
   -> val AUC=0.696326
  try cfg={'layers': 2, 'nhead': 4, 'ffn': 128} (epochs=1)
   -> val AUC=0.700610
AUTO-BST: best={'layers': 3, 'nhead': 8, 'ffn': 256} (val AUC=0.713384)
[5/9] Build final model ... backbone=bst, bst_cfg={'nhead': 8, 'num_layers': 3, 'dim_ff': 256}
[6/9] Train ...
Epoch 01 | Train 1.23508 | Val nan | AUC 0.733744 | PR-AUC 0.070817
Epoch 02 | Train 1.19515 | Val nan | AUC 0.73

In [1]:
# -*- coding: utf-8 -*-
"""
CTR Baseline — DCN-V2 + (DIEN or BST) with DIEN Auxiliary Next-Item Loss
[Notebook, OOM-safe, NaN-safe, Auto-tune aux_lambda/aux_neg_k]

- 시퀀스: 해시 임베딩(고정 버킷), Lazy 파싱
- 본체: Deep & Cross v2(CrossNetMix) + Deep tower
- 시퀀스 백본:
    * DIEN: GRU → DIN-activation attention + 보조손실(다음 아이템 InfoNCE)
    * BST : 얕은 Transformer + 위치임베딩 (보조손실 없음)
- 범주형: train 기준 factorize(OOV), 연속형: BatchNorm1d
- 불균형: pos_weight, 지표: AUC/PR-AUC + EarlyStopping
- 안정성: fp16 마스킹/정규화, all-PAD, NaN/±inf 정리
- 오토튜닝: DIEN용 aux_lambda / aux_neg_k를 소형 서브셋에서 1-epoch씩 빠르게 탐색
"""

import os, re, json, random, gc, math, warnings
warnings.filterwarnings("ignore")

import numpy as np
import pandas as pd

from sklearn.metrics import roc_auc_score, average_precision_score
from sklearn.model_selection import train_test_split

import torch
import torch.nn as nn
import torch.nn.functional as F
from torch.utils.data import Dataset, DataLoader
from torch.cuda.amp import autocast, GradScaler

# =============== Utils ===============
def set_seed(seed=42):
    random.seed(seed); os.environ["PYTHONHASHSEED"]=str(seed)
    np.random.seed(seed); torch.manual_seed(seed); torch.cuda.manual_seed_all(seed)
    torch.backends.cudnn.deterministic=True; torch.backends.cudnn.benchmark=False

def infer_device(arg="auto"):
    if arg=="cpu": return torch.device("cpu")
    if arg=="cuda" or (arg=="auto" and torch.cuda.is_available()):
        print("Using CUDA"); return torch.device("cuda")
    print("Using CPU"); return torch.device("cpu")

def parse_seq_string(s: str):
    if s is None: return []
    s = str(s).strip()
    if not s or s.lower()=="nan": return []
    if s.startswith("[") and s.endswith("]"):
        s = s[1:-1]; toks = [t.strip().strip("'\"") for t in s.split(",")]
    else:
        s = re.sub(r"[^\d]+", ",", s); toks = [t for t in s.split(",") if t]
    out=[]
    for t in toks:
        try: out.append(int(t))
        except Exception:
            try: out.append(int(float(t)))
            except Exception: pass
    return out

def emb_dim_from_card(card, cap=64):
    return int(min(cap, max(8, round(1.6*(card**0.25)))))

# =============== Config ===============
class Args:
    # Files
    train="./train.parquet"; test="./test.parquet"
    label="clicked"; seq_col="seq"; id_col="ID"

    # Data usage
    sample_subset=1.0
    max_seq_len=50
    HASH_BUCKETS=262_144         # 2^18
    seq_emb_dim=64
    dropout=0.2

    # Backbone: "dien" or "bst"
    seq_backbone="dien"
    # BST config
    bst_layers=2
    bst_nhead=4
    bst_ffn=128

    # DIEN Auxiliary Loss
    aux_lambda=0.2
    aux_neg_k=32

    # DCN-V2 (CrossNetMix)
    cross_layers=3
    cross_low_rank=32
    cross_num_experts=4

    # Deep tower
    deep_units=[512,256,128]

    # Training
    epochs=3; batch_size=512; lr=1e-3; patience=2
    seed=42; device="auto"; num_workers=0; pin_memory=False

    # Auto-tune (for DIEN)
    autotune_aux=True
    tune_fraction=0.05
    tune_epochs=1
    search_aux_lambda=[0.1, 0.2, 0.3]
    search_aux_negk=[16, 32, 64]

args=Args()
set_seed(args.seed); device=infer_device(args.device)

# =============== (옵션) 더미 데이터 생성 (파일 없을 때만) ===============
def create_dummy(num_rows, is_test=False):
    rng=np.random.default_rng(0)
    df=pd.DataFrame({
        "cont_feat1": rng.normal(size=num_rows),
        "cont_feat2": rng.random(num_rows)*10,
        "gender": rng.choice(["M","F","U"], num_rows, p=[0.45,0.45,0.1]),
        "age_group": rng.choice(["10s","20s","30s","40s+"], num_rows),
        "inventory_id": rng.integers(500, 520, num_rows),
        "day_of_week": rng.integers(0,7,num_rows),
        "hour": rng.integers(0,24,num_rows),
        "l_feat_14": rng.integers(1000,1050,num_rows),
    })
    seqs=[]
    for _ in range(num_rows):
        fmt=rng.choice(["pipe","comma","json","empty","single"])
        L=int(rng.integers(1,args.max_seq_len+10))
        items=[str(int(rng.integers(1000,1500))) for _ in range(L)]
        if fmt=="pipe": seqs.append("|".join(items))
        elif fmt=="comma": seqs.append(",".join(items))
        elif fmt=="json": seqs.append(f"[{','.join(items)}]")
        elif fmt=="empty": seqs.append(np.nan)
        else: seqs.append(items[0])
    df[args.seq_col]=seqs
    if is_test: df[args.id_col]=np.arange(num_rows)
    else: df[args.label]=rng.integers(0,2,num_rows)
    return df

if not os.path.exists(args.train):
    print("train.parquet 없음 → 더미 20k 생성"); create_dummy(20_000,False).to_parquet(args.train)
if not os.path.exists(args.test):
    print("test.parquet 없음 → 더미 5k 생성"); create_dummy(5_000,True).to_parquet(args.test)

# =============== Load & Prep ===============
print("[1/9] Load parquet ...")
train=pd.read_parquet(args.train); test=pd.read_parquet(args.test)
assert args.label in train.columns and args.seq_col in train.columns

if 0<args.sample_subset<1.0:
    print(f"Subsampling to {args.sample_subset*100:.1f}%")
    train=train.sample(frac=args.sample_subset, random_state=args.seed).reset_index(drop=True)

must_drop={args.label,args.seq_col,args.id_col}
base_cols=[c for c in train.columns if c not in must_drop]

# ---- 타깃 자동 선택 + 범주형/연속형 분리 ----
fixed_cats = [c for c in ["gender","age_group","inventory_id","day_of_week","hour"] if c in base_cols]
all_lfeats = [c for c in base_cols if c.startswith("l_feat_")]

preferred_targets = ["ad_id","creative_id","l_feat_14"]
present_targets = [c for c in preferred_targets if c in base_cols]
target_name = present_targets[0] if len(present_targets)>0 else None
if target_name is None and "l_feat_14" in all_lfeats:
    target_name = "l_feat_14"

CAT_CARD_MAX = 200_000
cand_cats = fixed_cats.copy()

tr_tmp, va_tmp = train_test_split(train, test_size=0.15, random_state=args.seed, stratify=train[args.label])
for c in all_lfeats:
    try:
        nunq = max(tr_tmp[c].nunique(dropna=True), va_tmp[c].nunique(dropna=True))
    except Exception:
        nunq = pd.concat([tr_tmp[c], va_tmp[c]], axis=0).nunique(dropna=True)
    if nunq <= CAT_CARD_MAX:
        cand_cats.append(c)

if target_name is not None and target_name not in cand_cats and target_name in base_cols:
    cand_cats.append(target_name)

cont_cols=[c for c in base_cols if c not in cand_cats]
print(f"[1.1] target_name={target_name}  cont={len(cont_cols)}  cat={len(cand_cats)}")

# 최종 split
tr,va=train_test_split(train, test_size=0.15, random_state=args.seed, stratify=train[args.label]); del train; gc.collect()

# 범주형 맵 (train 기준)
cat_maps, cat_cards = {}, {}
for c in cand_cats:
    cats=pd.Categorical(tr[c])
    cat_maps[c]={v:i for i,v in enumerate(cats.categories)}
    cat_cards[c]=len(cats.categories)

# 시퀀스 해시 임베딩
PAD_ID, OOV_ID, SEQ_BASE = 0, 1, 2
seq_vocab_size = args.HASH_BUCKETS + SEQ_BASE
def seq_to_ids_hash(lst, max_len=50):
    ids=[SEQ_BASE + (int(t)%args.HASH_BUCKETS) for t in lst][-max_len:]
    if len(ids)<max_len: ids=[PAD_ID]*(max_len-len(ids))+ids
    return np.array(ids, dtype=np.int32)

# =============== Dataset (Lazy seq parsing) ===============
class CTRDataset(Dataset):
    def __init__(self, df, cont_cols, cat_cols, cat_maps, seq_col, max_seq_len, label_col=None):
        self.df=df.reset_index(drop=True)
        self.cont_cols, self.cat_cols = cont_cols, cat_cols
        self.cat_maps, self.seq_col = cat_maps, seq_col
        self.max_seq_len=max_seq_len; self.has_label=label_col is not None; self.label_col=label_col
        self.Xc=self.df[self.cont_cols].astype(np.float32).fillna(0.0).values if self.cont_cols else None
        self.Xcats={c:self.df[c].map(self.cat_maps[c]).fillna(cat_cards[c]).astype(np.int64).values for c in self.cat_cols}
        if self.has_label: self.y=self.df[self.label_col].astype(np.float32).values
    def __len__(self): return len(self.df)
    def __getitem__(self, idx):
        xc=torch.from_numpy(self.Xc[idx]) if self.Xc is not None else torch.empty(0)
        cats={c: torch.tensor(self.Xcats[c][idx],dtype=torch.long) for c in self.cat_cols}
        lst=parse_seq_string(self.df.at[idx,self.seq_col])
        seq=torch.from_numpy(seq_to_ids_hash(lst, self.max_seq_len)).long()
        if self.has_label: return xc, cats, seq, torch.tensor(self.y[idx],dtype=torch.float32)
        return xc, cats, seq

# =============== DCN-V2 (CrossNetMix) ===============
class CrossLayerMix(nn.Module):
    def __init__(self, dim, low_rank=32, num_experts=4):
        super().__init__()
        self.num_experts=num_experts
        self.U=nn.Parameter(torch.randn(num_experts, dim, low_rank)*0.01)
        self.V=nn.Parameter(torch.randn(num_experts, low_rank, dim)*0.01)
        self.C=nn.Parameter(torch.zeros(num_experts, dim))
        self.gating=nn.Linear(dim, num_experts, bias=False)
        self.bias=nn.Parameter(torch.zeros(dim))
    def forward(self, x0, x):
        gate=torch.softmax(self.gating(x), dim=-1)   # (B,E)
        Xu=torch.einsum("bd,edr->ber", x, self.U)    # (B,E,r)
        Xuv=torch.einsum("ber,erd->bed", Xu, self.V) # (B,E,d)
        Xuv = Xuv + self.C                            # (B,E,d)
        mix=torch.einsum("be,bed->bd", gate, Xuv)    # (B,d)
        out=x0*mix + x + self.bias                   # cross + residual
        return out

class CrossNetMix(nn.Module):
    def __init__(self, dim, num_layers=3, low_rank=32, num_experts=4):
        super().__init__()
        self.layers=nn.ModuleList([CrossLayerMix(dim, low_rank, num_experts) for _ in range(num_layers)])
    def forward(self, x):
        x0=x; xl=x
        for layer in self.layers:
            xl=layer(x0, xl)
        return xl

# =============== DIN Activation Unit (공통 어텐션) ===============
class DINActivationUnit(nn.Module):
    """w_i = MLP([q, k_i, q-k_i, q*k_i])"""
    def __init__(self, dim_q, dim_k, hidden=[64,32], dropout=0.0):
        super().__init__()
        in_dim = dim_q + dim_k + dim_q + dim_k
        layers=[]
        for h in hidden:
            layers += [nn.Linear(in_dim, h), nn.ReLU(), nn.Dropout(dropout)]
            in_dim=h
        layers += [nn.Linear(in_dim, 1)]
        self.net=nn.Sequential(*layers)
    def forward(self, q, K):
        B,L,Dk = K.shape
        q_exp = q.unsqueeze(1).expand(-1,L,-1)
        feats = [q_exp, K, q_exp-K, q_exp*K]
        z = torch.cat(feats, dim=2)
        w = self.net(z).squeeze(2)    # (B,L)
        return w

# =============== Sequence Backbones ===============
class DIENBackbone(nn.Module):
    """GRU → DIN-activation attention  (+ Aux: next-item InfoNCE)"""
    def __init__(self, vocab_size, emb_dim, padding_idx=0, dropout=0.2,
                 pad_id=0, hash_base=2, hash_buckets=262_144, aux_neg_k=32):
        super().__init__()
        self.emb = nn.Embedding(vocab_size, emb_dim, padding_idx=padding_idx)
        self.gru = nn.GRU(input_size=emb_dim, hidden_size=emb_dim, batch_first=True)
        self.att = DINActivationUnit(emb_dim, emb_dim, hidden=[64,32], dropout=dropout)
        # Aux configs
        self.pad_id = pad_id
        self.hash_base = hash_base
        self.hash_buckets = hash_buckets
        self.aux_neg_k = aux_neg_k
        self.proj = nn.Linear(emb_dim, emb_dim, bias=False)  # for Q projection in InfoNCE

    def forward(self, seq_ids, q_vec):
        K0 = self.emb(seq_ids)                 # (B,L,D)
        H, _ = self.gru(K0)                    # (B,L,D)
        logits_w = self.att(q_vec, H)          # (B,L)
        maskL = (seq_ids != self.pad_id)
        # fp16-safe masking & all-PAD safe
        negv = torch.finfo(logits_w.dtype).min
        logits_w = logits_w.masked_fill(~maskL, negv)
        valid = maskL.any(dim=1, keepdim=True)
        maxv = torch.where(valid,
                           logits_w.max(dim=1, keepdim=True).values,
                           torch.zeros_like(logits_w[:, :1]))
        logits_w = torch.where(valid, logits_w - maxv, torch.zeros_like(logits_w))
        alpha = torch.where(valid, torch.softmax(logits_w, dim=1), torch.zeros_like(logits_w))
        alpha = torch.nan_to_num(alpha, nan=0.0)
        interest = torch.sum(H * alpha.unsqueeze(2), dim=1)   # (B,D)
        return interest, H

    @torch.no_grad()
    def _sample_negatives(self, num_pos: int, device):
        # Uniform negatives over hash space (allow duplicates; OK for contrastive)
        low = self.hash_base
        high = self.hash_base + self.hash_buckets
        # shape: (num_pos, K)
        neg = torch.randint(low, high, size=(num_pos, self.aux_neg_k), device=device)
        return neg

    def aux_next_item_loss(self, seq_ids, H):
        """
        InfoNCE over next-token:
          y_true = seq[:,1:], Q = proj(H[:,:-1,:])
          logits = concat( dot(Q, emb(y_true)), dot(Q, emb(negatives K)) )  -> CE(target=0)
        """
        y = seq_ids[:, 1:]                 # (B, L-1)
        Q = H[:, :-1, :]                   # (B, L-1, D)
        mask = (y != self.pad_id)          # (B, L-1)

        num_pos = int(mask.sum().item())
        if num_pos == 0:
            return H.new_tensor(0.0)

        # flatten valid positions
        y_flat = y[mask]                   # (N,)
        Q_flat = Q[mask]                   # (N, D)

        # negatives: (N, K)
        neg_ids = self._sample_negatives(num_pos=len(y_flat), device=y.device)

        # embeddings
        E_true = self.emb(y_flat)          # (N, D)
        E_neg  = self.emb(neg_ids)         # (N, K, D)

        # project queries
        Qp = self.proj(Q_flat)             # (N, D)

        # logits
        logit_true = (Qp * E_true).sum(dim=1, keepdim=True)              # (N,1)
        logit_neg  = torch.einsum("nd,nkd->nk", Qp, E_neg)               # (N,K)

        logits = torch.cat([logit_true, logit_neg], dim=1)               # (N, 1+K)
        targets = torch.zeros(len(y_flat), dtype=torch.long, device=y.device)  # index 0 is true

        loss = F.cross_entropy(logits, targets)
        return loss

class PositionalEncoding(nn.Module):
    def __init__(self, d_model, max_len=1024):
        super().__init__()
        pe = torch.zeros(max_len, d_model)
        position = torch.arange(0, max_len).unsqueeze(1)
        div_term = torch.exp(torch.arange(0, d_model, 2)*(-math.log(10000.0)/d_model))
        pe[:, 0::2] = torch.sin(position*div_term)
        pe[:, 1::2] = torch.cos(position*div_term)
        self.register_buffer('pe', pe.unsqueeze(0))  # (1,max_len,d)
    def forward(self, x):
        L = x.size(1)
        return x + self.pe[:, :L, :]

class BSTBackbone(nn.Module):
    """얕은 Transformer Encoder (2~4층) + PAD 마스크, 위치임베딩"""
    def __init__(self, vocab_size, emb_dim, nhead=4, num_layers=2, dim_ff=128, dropout=0.2, padding_idx=0):
        super().__init__()
        self.emb = nn.Embedding(vocab_size, emb_dim, padding_idx=padding_idx)
        self.pos = PositionalEncoding(emb_dim, max_len=2048)
        encoder_layer = nn.TransformerEncoderLayer(d_model=emb_dim, nhead=nhead, dim_feedforward=dim_ff,
                                                   batch_first=True, dropout=dropout, activation="relu", norm_first=True)
        self.enc = nn.TransformerEncoder(encoder_layer, num_layers=num_layers)
        self.att = DINActivationUnit(emb_dim, emb_dim, hidden=[64,32], dropout=dropout)
        self.pad_id = padding_idx
    def forward(self, seq_ids, q_vec):
        K0 = self.emb(seq_ids)
        K0 = self.pos(K0)
        key_pad_mask = (seq_ids == self.pad_id)
        H = self.enc(K0, src_key_padding_mask=key_pad_mask)
        logits_w = self.att(q_vec, H)
        negv = torch.finfo(logits_w.dtype).min
        logits_w = logits_w.masked_fill(key_pad_mask, negv)
        valid = (~key_pad_mask).any(dim=1, keepdim=True)
        maxv = torch.where(valid,
                           logits_w.max(dim=1, keepdim=True).values,
                           torch.zeros_like(logits_w[:, :1]))
        logits_w = torch.where(valid, logits_w - maxv, torch.zeros_like(logits_w))
        alpha = torch.where(valid, torch.softmax(logits_w, dim=1), torch.zeros_like(logits_w))
        alpha = torch.nan_to_num(alpha, nan=0.0)
        interest = torch.sum(H * alpha.unsqueeze(2), dim=1)
        return interest

# =============== Model: DCN-V2 + (DIEN or BST) ===============
class DCN_SEQ_Model(nn.Module):
    def __init__(self, cont_dim, cat_cards, seq_vocab_size, target_name=None,
                 seq_emb_dim=64, seq_backbone="dien",
                 bst_cfg=None, deep_units=[512,256,128],
                 cross_layers=3, cross_low_rank=32, cross_num_experts=4, dropout=0.2,
                 pad_id=0, hash_base=2, hash_buckets=262_144, aux_neg_k=32):
        super().__init__()
        self.has_cont=cont_dim>0
        if self.has_cont: self.bn=nn.BatchNorm1d(cont_dim)

        # categorical embeddings
        self.cat_embs=nn.ModuleDict()
        for name, card in cat_cards.items():
            dim=emb_dim_from_card(card+1)
            self.cat_embs[name]=nn.Embedding(card+1+1, dim)  # +1 OOV
        self.cat_total_dim = sum(emb.embedding_dim for emb in self.cat_embs.values())

        # base tabular x0
        self.tab_dim = (cont_dim if self.has_cont else 0) + self.cat_total_dim

        # DCN-V2
        self.cross = CrossNetMix(self.tab_dim, num_layers=cross_layers,
                                 low_rank=cross_low_rank, num_experts=cross_num_experts)

        # Deep tower
        deep_layers=[]; in_dim=self.tab_dim
        for h in deep_units:
            deep_layers += [nn.Linear(in_dim,h), nn.ReLU(), nn.Dropout(dropout)]
            in_dim=h
        self.deep = nn.Sequential(*deep_layers)
        self.deep_out_dim = deep_units[-1] if len(deep_units)>0 else self.tab_dim

        # Query(target) 임베딩
        self.target_name = target_name if (target_name in self.cat_embs) else None
        if self.target_name is not None:
            tdim = self.cat_embs[self.target_name].embedding_dim
        else:
            tdim = seq_emb_dim
        self.proj_t = nn.Linear(tdim, seq_emb_dim, bias=False) if tdim!=seq_emb_dim else nn.Identity()

        # Sequence backbone
        self.seq_backbone = seq_backbone.lower()
        if self.seq_backbone == "bst":
            cfg = bst_cfg or {"nhead":4, "num_layers":2, "dim_ff":128}
            self.seq_enc = BSTBackbone(seq_vocab_size, seq_emb_dim,
                                       nhead=cfg["nhead"], num_layers=cfg["num_layers"], dim_ff=cfg["dim_ff"],
                                       dropout=dropout, padding_idx=pad_id)
            self.dien_aux = False
        else:
            self.seq_enc = DIENBackbone(seq_vocab_size, seq_emb_dim, padding_idx=pad_id, dropout=dropout,
                                        pad_id=pad_id, hash_base=hash_base, hash_buckets=hash_buckets, aux_neg_k=aux_neg_k)
            self.dien_aux = True

        # Final head: [cross_out, deep_out, interest_vec]
        final_in = self.tab_dim + self.deep_out_dim + seq_emb_dim
        self.head = nn.Sequential(
            nn.Linear(final_in, 256), nn.ReLU(), nn.Dropout(dropout),
            nn.Linear(256, 1)
        )

        self.seq_emb_dim = seq_emb_dim  # for fallback q zeros

    def forward(self, xc, xcats, seq_ids):
        # ----- Tabular -----
        outs=[]
        if self.has_cont: outs.append(self.bn(xc))
        if len(self.cat_embs)>0:
            outs.append(torch.cat([self.cat_embs[n](xcats[n]) for n in self.cat_embs], dim=1))
        x0 = torch.cat(outs, dim=1) if len(outs)>1 else outs[0]   # (B, tab_dim)

        # DCN-V2 cross / Deep tower
        x_cross = self.cross(x0)                                  # (B, tab_dim)
        x_deep  = self.deep(x0)                                   # (B, deep_dim)

        # Query(target) vector
        if self.target_name is not None:
            q = self.cat_embs[self.target_name](xcats[self.target_name])  # (B,Dt)
            q = self.proj_t(q)                                            # (B,D)
        else:
            q = torch.zeros(seq_ids.size(0), self.seq_emb_dim, device=seq_ids.device)

        # Sequence → interest vector (+ optional aux loss)
        if self.dien_aux:
            interest, H = self.seq_enc(seq_ids, q)                # (B,D), (B,L,D)
            aux_loss = self.seq_enc.aux_next_item_loss(seq_ids, H)
        else:
            interest = self.seq_enc(seq_ids, q)
            aux_loss = None

        # 출력 결합
        z = torch.cat([x_cross, x_deep, interest], dim=1)
        logits = self.head(z).squeeze(1)
        return logits, aux_loss

# =============== Data & Loaders ===============
ds_tr=CTRDataset(tr, cont_cols, cand_cats, cat_maps, args.seq_col, args.max_seq_len, label_col=args.label)
ds_va=CTRDataset(va, cont_cols, cand_cats, cat_maps, args.seq_col, args.max_seq_len, label_col=args.label)
ds_te=CTRDataset(test, cont_cols, cand_cats, cat_maps, args.seq_col, args.max_seq_len, label_col=None)

n_pos=float((tr[args.label]==1).sum()); n_neg=float(len(tr)-n_pos)
pos_weight=torch.tensor(max(1.0, n_neg/max(1.0,n_pos)), dtype=torch.float32, device=device)
print(f"[2/9] Train label: pos={int(n_pos)} neg={int(n_neg)}  pos_weight={pos_weight.item():.3f}")

dl_tr=DataLoader(ds_tr, batch_size=args.batch_size, shuffle=True,
                 num_workers=args.num_workers, pin_memory=args.pin_memory)
dl_va=DataLoader(ds_va, batch_size=args.batch_size, shuffle=False,
                 num_workers=args.num_workers, pin_memory=args.pin_memory)
dl_te=DataLoader(ds_te, batch_size=args.batch_size, shuffle=False,
                 num_workers=args.num_workers, pin_memory=args.pin_memory)

# =============== Auto-tune aux_lambda/aux_neg_k (DIEN only) ===============
def build_model_with(aux_lambda, aux_neg_k):
    bst_cfg={"nhead":args.bst_nhead, "num_layers":args.bst_layers, "dim_ff":args.bst_ffn}
    m = DCN_SEQ_Model(
        cont_dim=len(cont_cols), cat_cards=cat_cards, seq_vocab_size=seq_vocab_size,
        target_name=target_name, seq_emb_dim=args.seq_emb_dim, seq_backbone=args.seq_backbone,
        bst_cfg=bst_cfg, deep_units=args.deep_units,
        cross_layers=args.cross_layers, cross_low_rank=args.cross_low_rank, cross_num_experts=args.cross_num_experts,
        dropout=args.dropout, pad_id=PAD_ID, hash_base=SEQ_BASE, hash_buckets=args.HASH_BUCKETS, aux_neg_k=aux_neg_k
    ).to(device)
    return m

def make_small_loader(ds, frac, shuffle):
    n = max(1, int(len(ds)*frac))
    idx = torch.randperm(len(ds))[:n].tolist() if shuffle else list(range(n))
    class _Wrap(Dataset):
        def __init__(self, base, idxs): self.base, self.idxs = base, idxs
        def __len__(self): return len(self.idxs)
        def __getitem__(self, i): return self.base[self.idxs[i]]
    return DataLoader(_Wrap(ds, idx), batch_size=args.batch_size, shuffle=shuffle,
                      num_workers=0, pin_memory=False)

def quick_eval(aux_lambda, aux_neg_k):
    model = build_model_with(aux_lambda, aux_neg_k)
    criterion = nn.BCEWithLogitsLoss(pos_weight=pos_weight)
    optimizer = torch.optim.AdamW(model.parameters(), lr=args.lr, weight_decay=1e-4)
    scaler = GradScaler(enabled=(device.type=="cuda"))
    dl_tr_small = make_small_loader(ds_tr, args.tune_fraction, shuffle=True)
    dl_va_small = make_small_loader(ds_va, args.tune_fraction, shuffle=False)

    model.train()
    for xc, cats, seq, y in dl_tr_small:
        xc, seq, y = xc.to(device, non_blocking=True), seq.to(device, non_blocking=True), y.to(device, non_blocking=True)
        cats_dev = {k: v.to(device, non_blocking=True) for k, v in cats.items()}
        optimizer.zero_grad(set_to_none=True)
        with autocast(enabled=(device.type=="cuda")):
            logits, aux_loss = model(xc, cats_dev, seq)
            main_loss = criterion(logits, y)
            loss = main_loss + (aux_lambda*aux_loss if aux_loss is not None else 0.0)
        scaler.scale(loss).backward(); scaler.step(optimizer); scaler.update()

    model.eval(); ys=[]; ps=[]
    with torch.no_grad():
        for xc, cats, seq, y in dl_va_small:
            xc, seq = xc.to(device, non_blocking=True), seq.to(device, non_blocking=True)
            cats_dev = {k: v.to(device, non_blocking=True) for k, v in cats.items()}
            with autocast(enabled=(device.type=="cuda")):
                logits, _ = model(xc, cats_dev, seq)
                prob = torch.sigmoid(logits)
            p = prob.detach().cpu().numpy().astype(np.float64)
            p = np.nan_to_num(p, nan=0.0, posinf=1.0, neginf=0.0)
            ys.append(y.numpy()); ps.append(p)
    y_true = np.concatenate(ys); y_prob = np.concatenate(ps)
    auc = roc_auc_score(y_true, y_prob)
    return float(auc)

best_aux_lambda, best_aux_negk, best_auc = args.aux_lambda, args.aux_neg_k, -1.0

if args.autotune_aux and args.seq_backbone.lower()=="dien":
    print(f"[3/9] AUTOTUNE: {int(args.tune_fraction*100)}% subset, {args.tune_epochs} epoch per combo")
    tried = 0
    for lam in args.search_aux_lambda:
        for negk in args.search_aux_negk:
            tried += 1
            try:
                # local variable for quick_eval closure
                aux_lambda = lam
                auc = quick_eval(lam, negk)
                print(f"  trial #{tried:02d}: aux_lambda={lam:.3f}, aux_neg_k={negk} -> AUC={auc:.6f}")
                if auc > best_auc:
                    best_auc, best_aux_lambda, best_aux_negk = auc, lam, negk
            except Exception as e:
                print(f"  trial #{tried:02d} failed ({lam}, {negk}): {e}")
    if best_auc >= 0:
        args.aux_lambda = best_aux_lambda
        args.aux_neg_k  = best_aux_negk
        print(f"[3.1/9] AUTOTUNE best -> aux_lambda={args.aux_lambda:.3f}, aux_neg_k={args.aux_neg_k}, AUC={best_auc:.6f}")
else:
    print("[3/9] AUTOTUNE: skipped (disabled or backbone != DIEN)")

# =============== Build final model ===============
print("[4/9] Build final model ...")
bst_cfg={"nhead":args.bst_nhead, "num_layers":args.bst_layers, "dim_ff":args.bst_ffn}
model=DCN_SEQ_Model(
    cont_dim=len(cont_cols), cat_cards=cat_cards, seq_vocab_size=seq_vocab_size,
    target_name=target_name, seq_emb_dim=args.seq_emb_dim, seq_backbone=args.seq_backbone,
    bst_cfg=bst_cfg, deep_units=args.deep_units,
    cross_layers=args.cross_layers, cross_low_rank=args.cross_low_rank, cross_num_experts=args.cross_num_experts,
    dropout=args.dropout, pad_id=PAD_ID, hash_base=SEQ_BASE, hash_buckets=args.HASH_BUCKETS, aux_neg_k=args.aux_neg_k
).to(device)

criterion=nn.BCEWithLogitsLoss(pos_weight=pos_weight)
optimizer=torch.optim.AdamW(model.parameters(), lr=args.lr, weight_decay=1e-4)
scheduler=torch.optim.lr_scheduler.CosineAnnealingLR(optimizer, T_max=args.epochs)
scaler=GradScaler(enabled=(device.type=="cuda"))

# =============== Train ===============
best_auc, best_state, wait=-1.0, None, 0
final_epoch, final_prauc=0, 0.0

print(f"[5/9] Train ... (backbone={args.seq_backbone}, aux_lambda={args.aux_lambda}, aux_neg_k={args.aux_neg_k})")
for epoch in range(1, args.epochs+1):
    model.train(); tr_loss=0.0
    for xc, cats, seq, y in dl_tr:
        xc, seq, y = xc.to(device, non_blocking=True), seq.to(device, non_blocking=True), y.to(device, non_blocking=True)
        cats_dev={k:v.to(device, non_blocking=True) for k,v in cats.items()}
        optimizer.zero_grad(set_to_none=True)
        with autocast(enabled=(device.type=="cuda")):
            logits, aux = model(xc, cats_dev, seq)
            main_loss = criterion(logits, y)
            loss = main_loss + (args.aux_lambda*aux if aux is not None else 0.0)
        scaler.scale(loss).backward(); scaler.step(optimizer); scaler.update()
        tr_loss += loss.item()*y.size(0)
    tr_loss/=len(ds_tr); scheduler.step()

    # ----- Validation -----
    model.eval(); va_loss=0.0; ys=[]; ps=[]
    with torch.no_grad():
        for xc, cats, seq, y in dl_va:
            xc, seq = xc.to(device, non_blocking=True), seq.to(device, non_blocking=True)
            cats_dev={k:v.to(device, non_blocking=True) for k,v in cats.items()}
            with autocast(enabled=(device.type=="cuda")):
                logits, aux = model(xc, cats_dev, seq)
                main_loss = criterion(logits, y.to(device))
                loss = main_loss + (args.aux_lambda*aux if aux is not None else 0.0)
                prob=torch.sigmoid(logits)
            p = prob.detach().cpu().numpy().astype(np.float64)
            p = np.nan_to_num(p, nan=0.0, posinf=1.0, neginf=0.0)
            va_loss += loss.item()*len(y)
            ys.append(y.numpy()); ps.append(p)
    va_loss/=len(ds_va)
    y_true=np.concatenate(ys); y_prob=np.concatenate(ps)
    auc=roc_auc_score(y_true, y_prob); prauc=average_precision_score(y_true, y_prob)
    print(f"Epoch {epoch:02d} | Train {tr_loss:.5f} | Val {va_loss:.5f} | AUC {auc:.6f} | PR-AUC {prauc:.6f}")

    if auc > best_auc + 1e-5:
        best_auc, final_prauc, final_epoch = auc, prauc, epoch
        best_state={k:v.cpu().clone() for k,v in model.state_dict().items()}; wait=0
    else:
        wait += 1
        if wait >= args.patience:
            print(f"Early stopping. Best AUC={best_auc:.6f} at epoch {final_epoch}."); break

if best_state is not None:
    model.load_state_dict(best_state)

# =============== Inference ===============
print("[6/9] Predict test ...")
model.eval(); probs=[]
with torch.no_grad():
    for xc, cats, seq in dl_te:
        xc, seq = xc.to(device, non_blocking=True), seq.to(device, non_blocking=True)
        cats_dev={k:v.to(device, non_blocking=True) for k,v in cats.items()}
        with autocast(enabled=(device.type=="cuda")):
            logits, _ = model(xc, cats_dev, seq)
            prob=torch.sigmoid(logits)
        p = prob.detach().cpu().numpy().astype(np.float64)
        p = np.nan_to_num(p, nan=0.0, posinf=1.0, neginf=0.0)
        probs.append(p)
probs=np.concatenate(probs)

# =============== Save ===============
print("[7/9] Save outputs ...]")
sub_path=f"./dcnv2_{args.seq_backbone}_aux_submit.csv"
meta_path=f"./dcnv2_{args.seq_backbone}_aux_meta.json"
pd.DataFrame({args.id_col: test[args.id_col].values, "clicked": probs}).to_csv(sub_path, index=False)

# 최종 메타
print("[8/9] Save meta ...")
meta={
    "columns":{"continuous":cont_cols, "categorical":cand_cats},
    "seq_vocab":{"type":"hash","buckets":args.HASH_BUCKETS,"pad_id":0,"oov_id":1},
    "hyperparameters":{
        "sample_subset":args.sample_subset,"max_seq_len":args.max_seq_len,
        "seq_emb_dim":args.seq_emb_dim,"dropout":args.dropout,
        "epochs":args.epochs,"batch_size":args.batch_size,"lr":args.lr,
        "cross_layers":args.cross_layers,"cross_low_rank":args.cross_low_rank,"cross_num_experts":args.cross_num_experts,
        "deep_units":args.deep_units, "seq_backbone":args.seq_backbone,
        "bst_layers":args.bst_layers,"bst_nhead":args.bst_nhead,"bst_ffn":args.bst_ffn,
        "aux_lambda":args.aux_lambda, "aux_neg_k":args.aux_neg_k,
        "autotuned": bool(args.autotune_aux and args.seq_backbone.lower()=="dien")
    },
    "performance":{"best_epoch":int(final_epoch),"AUC":float(best_auc),"PR_AUC":float(final_prauc)},
    "target_name": target_name
}
with open(meta_path,"w",encoding="utf-8") as f: json.dump(meta,f,ensure_ascii=False,indent=2)

print(f"[9/9] Done. \n - {sub_path}\n - {meta_path}")


Using CUDA
[1/9] Load parquet ...
[1.1] target_name=l_feat_14  cont=85  cat=32
[2/9] Train label: pos=173552 neg=8925000  pos_weight=51.426
[3/9] AUTOTUNE: 5% subset, 1 epoch per combo
  trial #01: aux_lambda=0.100, aux_neg_k=16 -> AUC=0.719335
  trial #02: aux_lambda=0.100, aux_neg_k=32 -> AUC=0.714749
  trial #03: aux_lambda=0.100, aux_neg_k=64 -> AUC=0.717725
  trial #04: aux_lambda=0.200, aux_neg_k=16 -> AUC=0.715098
  trial #05: aux_lambda=0.200, aux_neg_k=32 -> AUC=0.715779
  trial #06: aux_lambda=0.200, aux_neg_k=64 -> AUC=0.724423
  trial #07: aux_lambda=0.300, aux_neg_k=16 -> AUC=0.712248
  trial #08: aux_lambda=0.300, aux_neg_k=32 -> AUC=0.684937
  trial #09: aux_lambda=0.300, aux_neg_k=64 -> AUC=0.714038
[3.1/9] AUTOTUNE best -> aux_lambda=0.200, aux_neg_k=64, AUC=0.724423
[4/9] Build final model ...
[5/9] Train ... (backbone=dien, aux_lambda=0.2, aux_neg_k=64)
Epoch 01 | Train 1.21255 | Val nan | AUC 0.734914 | PR-AUC 0.069982
Epoch 02 | Train 1.18869 | Val nan | AUC 0.7393